In [ ]:
%load_ext watermark


In [ ]:
from IPython.display import display, HTML
from matplotlib import pyplot as plt
import pandas as pd
import polars as pl
import seaborn as sns
from slugify import slugify
from teeplot import teeplot as tp
from tqdm import tqdm

from pylib._percentilestatcat_plot import (
    percentilestatcat_plot,
)
from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-05-19-compscreen-summary"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


In [ ]:
import itertools as it

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as scipy_stats
import seaborn as sns


def _bootstrap_p_value(
    grp: pd.Series,
    threshold: float,
    n_boot: int = 100_000,
    max_mem_bytes: int = 1024**3,
) -> float:
    n = len(grp)

    bytes_per_resample = n * grp.dtype.itemsize
    batch_size = max(1, min(n_boot, int(max_mem_bytes // bytes_per_resample)))

    successes = 0
    for begin, end in it.pairwise(
        range(0, n_boot + batch_size - 1, batch_size)
    ):
        end = min(end, n_boot)
        means = np.random.choice(
            grp, size=(end - begin, n), replace=True
        ).mean(axis=1)
        successes += np.sum(means >= threshold)

    return successes / n_boot


def calc_stats(data: pd.DataFrame, x: str, y: str) -> plt.Axes:

    data = data.dropna(subset=y)

    threshold = 50

    n_boot = 10_000
    null_p = (data.loc[(data[y] != threshold), y] < threshold).mean()
    if np.isnan(null_p):
        null_p = 0.5

    results = []
    for label, grp in {
        "all": data[y],
        **{k: v[y] for k, v in data.groupby(x, observed=False)},
    }.items():

        # --- bootstrap test for mean < threshold ---
        p_boot = _bootstrap_p_value(grp, threshold, n_boot=n_boot)

        # --- binomial/sign test for median < threshold ---
        binom_k = np.sum(grp < threshold)
        binom_n = np.sum(grp != threshold & ~np.isnan(grp))
        p_binom = scipy_stats.binomtest(
            binom_k, binom_n, p=null_p, alternative="greater"
        ).pvalue

        # --- mann-whitney test for difference in medians ---
        if label != "all":
            focal = data.loc[data[x] == label, y].dropna()
            nonfocal = data.loc[data[x] != label, y].dropna()
            mw_u, mw_p = scipy_stats.mannwhitneyu(
                focal, nonfocal, alternative="greater"
            )
            cliffs_delta = 2 * mw_u / (len(focal) * len(nonfocal)) - 1
        else:
            mw_u, mw_p, cliffs_delta = np.nan, np.nan, np.nan

        results.append(
            {
                "label": label,
                "binom_n": binom_n,
                "binom_k": binom_k,
                "n": len(data),
                "n_boot": n_boot,
                "p_boot": p_boot,
                "p_binom": p_binom,
                "p_mw": mw_p,
                "u_mw": mw_u,
                "cliffs_delta": cliffs_delta,
                "na": np.sum(grp.isna()),
            },
        )

    return results


## Get Data


In [ ]:
data_sources = {
    "uk": "https://osf.io/4asyw/download",
    # "multistrain": "https://osf.io/u6ta9/download",
    # "vanilla": "https://osf.io/aysxt/download",
    # "vanilla-big": "https://osf.io/ry5nd/download",
}


In [ ]:
results = []

for source_name, url in tqdm(data_sources.items()):
    print(f"Downloading {source_name} data from {url}")
    df = pl.scan_parquet(
        url,
        low_memory=True,
        retries=5,
    )

    groupby_items = [
        *df.collect().group_by(
            ["trt_name", "trt_hsurf_bits", "replicate_uuid"],
        )
    ]
    for (trt_name, trt_hsurf_bits, replicate_uuid), group in tqdm(
        groupby_items,
    ):
        for y in (
            "defmut_norm_all-num_leaves",
            # "defmut_norm_ot_bin:week-num_leaves",
            # "defmut_norm_ot_bin:fortnight-num_leaves",
            # "defmut_norm_ot_bin:month-num_leaves",
            "defmut_norm_ot_bin:fortnight-num_leaves",
            # "defmut_norm_ot_bin:year-num_leaves",
            # "defmut_norm_match:variant_flavor-num_leaves",
            "defmut_norm_all-clade_duration",
            # "defmut_norm_ot_bin:week-clade_duration",
            # "defmut_norm_ot_bin:fortnight-clade_duration",
            # "defmut_norm_ot_bin:month-clade_duration",
            "defmut_norm_ot_bin:fortnight-clade_duration",
            # "defmut_norm_ot_bin:year-clade_duration",
            # "defmut_norm_match:variant_flavor-clade_duration",
        ):
            res = calc_stats(
                group.to_pandas(),
                x="is_focal_defmut",
                y=y,
            )
            results.extend(
                {
                    "source_name": source_name,
                    "trt_name": trt_name,
                    "trt_hsurf_bits": trt_hsurf_bits,
                    "replicate_uuid": replicate_uuid,
                    "y": y,
                    **record,
                }
                for record in res
            )


In [ ]:
results_df = pd.DataFrame(results)
results_df


In [ ]:
results_df["y"] = results_df["y"].str.removeprefix("defmut_norm_all-")
results_df["y"] = results_df["y"].str.removeprefix("defmut_norm_ot_bin:")
results_df["trt_name_"] = results_df["trt_name"].str.replace("/", "\n")


In [ ]:
for y in "p_binom", "p_boot", "p_mw", "u_mw", "cliffs_delta":
    with tp.teed(
        sns.catplot,
        data=results_df[results_df["trt_hsurf_bits"] == 0],
        y=y,
        x="y",
        col="trt_name_",
        row="label",
        hue="y",
        legend=False,
        kind="swarm",
        dodge=True,
        margin_titles=True,
        alpha=0.5,
        height=1.5,
        aspect=1.2,
        size=4,
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_titles(col_template="{col_name}", row_template="{row_name}")
        for ax in teed.axes.flat:
            ax.set_yscale("log")
            ax.axhline(0.5, color="black", linestyle=":")
            ax.axhline(0.05, color="red", linestyle="--")
            ax.tick_params(axis="x", rotation=-90)

        teed.tight_layout()
